# (6) Cilantro-SL classification and uncertainty quantification

In [ ]:
import pandas as pd
import pickle as pkl
import sys

sys.path.append("../")
sys.path.append("../nn_helpers")

import nn_helpers.via_film as via_film
import nn_helpers.pair_classifier as pc_module
import nn_helpers.training_framework as training_framework

path = "../outputs/gf_12L_30M_i2048_SL/generated_df"
name = "gene2vec_5x"
cv = 1
metrics_dict_path = f"../data/ablations/cv{cv}/{name}.pkl"

print(f"Running general model with FiLM pretraining using EXP5 CV{cv}")

print("Viability embedding generation")
df = pd.read_hdf(f"{path}/gene2vec_emb_mat.h5", "table")
print(f"df shape is {df.shape}")

Torch imports completed
INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Running general model with FiLM pretraining using EXP5 CV1
Viability embedding generation
df shape is (2172399, 641)


In [ ]:
d = pd.read_hdf(
    "../outputs/gf_12L_30M_i2048_SL/generated_df/gene2vec_5x_all_double.h5", "table"
)

pc = pc_module.pair_classifier(d, model_type=pc_module.ModelType.COMBINED)
framework = training_framework.Framework(None, pc, metrics_path=metrics_dict_path)

nn_save_paths = [f"../data/temp/nn_path_{i}.pth" for i in range(5)]

test, train = pc.setup_cv(5, cv=cv)

with open ("../data/temp/split_5x_cv1.pkl", "wb") as f:
    pkl.dump((test, train), f)

framework.run_cv(test, train, cv=cv, nn_save_paths=nn_save_paths)

In [ ]:
framework.all_test = test
framework.all_train = train
framework.folds = 1

def extract_name(x):
    return x["patient"]

net_list = [f"../data/temp/nn_path_{i}.pth" for i in range(5)]

framework.uncertainty_quantification(
    training_framework.UQ.MONDRIAN_CONFORMAL,
    net=net_list ,
    mondrian_class_dict={},
    extract_name=extract_name
)